# Re-run video processing

We re-run processing over all files listed in the files_with_proboscis.csv.

We use the _process_video function from choice_assay_pose_processor.py to run two ML models over each video:
- firstly we grab the first frame as an image and run an ML model to find the location of the feeding tubes
- secondly we run an ML model over every frame to identify the location of any bee present (max of 1)


In [1]:
import platform
from pathlib import Path
import pandas as pd
from choice_assay.etl.rerun_processing import run_rerun_processing

Logging expidite to default file: C:\Users\bee-ops\AppData\Local\Temp\expidite\20260716T150549459\logs\default_20260716T150549472.log at level 20
2026-07-16 17:05:49,474 expidite INFO   [4840] Loading C:\Users\bee-ops\.expidite\system.cfg...
Logging choice_assay to default file: C:\Users\bee-ops\AppData\Local\Temp\expidite\20260716T150549459\logs\default_20260716T150549472.log at level 20


In [2]:
# Required config
CONTAINER_NAME = "expidite-choiceassay-trapcam"

# Decide if we're running on Windows or Linux, and set the NAS root path accordingly
running_on_linux = "Linux" in platform.platform()

if running_on_linux:
    nas_root = Path("/bee-ops-disk/")
else:
    nas_root = Path("B://")

    # If we don't have access to the B: drive fall back to a local alternative
    if not nas_root.exists():
        print("Warning: B: drive not found, falling back to local alternative")
        nas_root = Path.home() / "bee-ops-disk"

DOWNLOAD_DIR = nas_root / "azure" / "choice_assay" / "expidite-choiceassay-trapcam"
OUTPUT_DIR = nas_root / "results" / "choice_assay_rerun"

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Container: {CONTAINER_NAME}")
print(f"Download directory: {DOWNLOAD_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

Container: expidite-choiceassay-trapcam
Download directory: C:\Users\bee-ops\bee-ops-disk\azure\choice_assay\expidite-choiceassay-trapcam
Output directory: C:\Users\bee-ops\bee-ops-disk\results\choice_assay_rerun


In [3]:
# Create a CSV file of all the files in the container.  About 3m for 80k files.
"""
files = DOWNLOAD_DIR.glob(f"{PREFIX}*{SUFFIX}")

# Save the files list to file
files_df = pd.DataFrame([str(f) for f in files], columns=["filename"])
files_df.to_csv("files_list.csv", index=False)
"""

'\nfiles = DOWNLOAD_DIR.glob(f"{PREFIX}*{SUFFIX}")\n\n# Save the files list to file\nfiles_df = pd.DataFrame([str(f) for f in files], columns=["filename"])\nfiles_df.to_csv("files_list.csv", index=False)\n'

In [4]:
# Run ML processing over all the video files listed in files_with_proboscis.csv
summary = run_rerun_processing(
    files_to_process=Path("files_with_proboscis.csv"),
    video_src_dir=DOWNLOAD_DIR,
    output_dir=OUTPUT_DIR,
)

print("Run summary:")
print(summary)

Found 13909 new videos to process. 21 already handled.
JIT local cache directory: C:\Users\bee-ops\AppData\Local\Temp\choice_assay_video_cache
Prefetch ahead: 6 files
Processing log: C:\Users\bee-ops\bee-ops-disk\results\choice_assay_rerun\processed_videos_log.csv
2026-07-16 17:06:13,117 choice_assay INFO   [4840] Pose model diagnostics: ultralytics=8.4.21 model_path=C:\Users\bee-ops\code\ChoiceAssay\src\choice_assay\resources\bee_best.pt names_count=1 names_keys=[0]
2026-07-16 17:06:46,935 choice_assay INFO   [4840] Pose timings: video=C:\Users\bee-ops\AppData\Local\Temp\choice_assay_video_cache\V3_CAVIDEO_d83add1a11c5_00_00_20260617T150555552_20260617T150815551.mp4 model_load=0.145s tube_detect=0.227s predict_setup=0.026s stream_iter=33.561s dataframe=0.003s markup_save=0.000s total=33.962s frames=701 rows=105 rows_per_frame=0.150
1/13909 @ 34.0 secs/video | infer=33.97s | copy=0.00s (3.7MB) | avg_copy=0.00s | avg_wait=0.00s | avg_infer=33.97s | ok=1 | miss_prefetch=0 | miss_process=

KeyboardInterrupt: 

In [ ]:
csv_paths = sorted(OUTPUT_DIR.rglob("*.csv"))
print(f"CSV files available locally: {len(csv_paths)}")

df_list = []
for csv_path in csv_paths:
    df = pd.read_csv(csv_path)
    if not df.empty:
        df["source_file"] = csv_path.name
        df_list.append(df)

if df_list:
    aggregated_df = pd.concat(df_list, ignore_index=True)
else:
    aggregated_df = pd.DataFrame()

print(f"Aggregated rows: {len(aggregated_df)}")
aggregated_df.head()

In [ ]:
# Data validation before behaviour classification
required_columns = ["Tube_prob_likelihood", "End_prob_likelihood"]
missing_columns = [col for col in required_columns if col not in aggregated_df.columns]

if aggregated_df.empty:
    print("Validation: aggregated_df is empty.")
elif missing_columns:
    msg = f"Validation failed. Missing required columns: {missing_columns}"
    raise KeyError(msg)
else:
    for col in required_columns:
        aggregated_df[col] = pd.to_numeric(aggregated_df[col], errors="coerce")

    invalid_rows = aggregated_df[required_columns].isna().any(axis=1).sum()
    print(f"Validation: {len(aggregated_df)} total rows")
    print(f"Validation: {invalid_rows} rows have invalid/missing likelihood values")

    if invalid_rows:
        print(
            aggregated_df.loc[
                aggregated_df[required_columns].isna().any(axis=1), [*required_columns, "source_file"]
            ].head()
        )

In [ ]:
# Define the function to classify behavior
def get_behaviour(row: pd.Series) -> str:
    behaviour = "No_prob"
    if (row["Tube_prob_likelihood"] >= 0.6) & (row["End_prob_likelihood"] >= 0.6):
        behaviour = "Drinking"
    elif (row["Tube_prob_likelihood"] >= 0.6) ^ (row["End_prob_likelihood"] >= 0.6):
        behaviour = "Prob_out"
    return behaviour


assert aggregated_df is not None, "aggregated_df should be defined at this point"
required_columns = ["Tube_prob_likelihood", "End_prob_likelihood"]
missing_columns = [col for col in required_columns if col not in aggregated_df.columns]

if aggregated_df.empty:
    print("No data loaded; behaviour classification skipped.")
elif missing_columns:
    msg = f"Missing required columns for behaviour classification: {missing_columns}"
    raise KeyError(msg)
else:
    before_count = len(aggregated_df)
    clean_df = aggregated_df.dropna(subset=required_columns).copy()
    dropped_count = before_count - len(clean_df)

    if dropped_count:
        print(f"Dropped {dropped_count} rows with invalid/missing likelihood values before classification.")

    clean_df["Behaviour"] = clean_df.apply(get_behaviour, axis=1)
    aggregated_df = clean_df
    print(aggregated_df["Behaviour"].value_counts(dropna=False))

aggregated_df.head()

In [ ]:
output_path = DOWNLOAD_DIR / "aggregated_journals_with_behaviour.csv"
if not aggregated_df.empty:
    aggregated_df.to_csv(output_path, index=False)
    print(f"Saved aggregated dataset to: {output_path.resolve()}")
else:
    print("Aggregated dataframe is empty; no output written.")